# Reto: copiloto de documentos técnicos

El copiloto recibe una pregunta en español, encuentra las páginas relevantes entre 300 páginas de reportes técnicos de energía y transporte (en inglés, con tablas y gráficas), las lee con un modelo de visión y lenguaje (VLM) y responde citando documento y página. Si la respuesta no está en la colección, lo dice.

Esta libreta solo carga las páginas, las preguntas de práctica y el modelo que lee las páginas. La búsqueda, la comparación de recuperadores, la abstención y la interfaz las construye el equipo. La carga de datos funciona en CPU.

In [ ]:
!pip install -q google-genai gradio

import io
import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq
import torch
from huggingface_hub import hf_hub_download
from PIL import Image

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", dispositivo)

## Páginas y preguntas de práctica

- `abrir(page_id)` devuelve la imagen (PIL) de una página; `paginas` guarda los JPEG para no llenar la memoria. El identificador tiene la forma `<documento>_p<página>`, por ejemplo `04a016c8_p841`, y es lo que el copiloto debe citar.
- `corpus` lista las 300 páginas con su documento y número de página.
- `practica` tiene 20 preguntas en español (`pregunta`), su versión original en inglés (`pregunta_original_en`), la respuesta de referencia (`respuesta`) y la página correcta (`page_id`).

Las páginas provienen de `vidore/syntheticDocQA_energy_test` (licencia MIT).

In [ ]:
URL_BASE = "https://raw.githubusercontent.com/noe-tec/machine-learning/main/datasets/docs_energia"
corpus = pd.read_csv(f"{URL_BASE}/corpus_paginas.csv")
practica = pd.read_csv(f"{URL_BASE}/preguntas_practica.csv")
assert len(corpus) == 300 and corpus.page_id.is_unique, f"Se esperaban 300 páginas únicas; hay {len(corpus)}."
assert len(practica) == 20, f"Se esperaban 20 preguntas de práctica; hay {len(practica)}."

ruta_parquet = hf_hub_download("vidore/syntheticDocQA_energy_test",
                               "data/test-00000-of-00001.parquet", repo_type="dataset")
tabla = pq.read_table(ruta_parquet, columns=["image_filename", "image"])
buscadas = dict(zip(corpus.image_filename, corpus.page_id))
paginas = {}
for lote in tabla.to_batches(64):
    for nombre, img in zip(lote.column(0).to_pylist(), lote.column(1).to_pylist()):
        if nombre in buscadas and buscadas[nombre] not in paginas:
            paginas[buscadas[nombre]] = img["bytes"]      # se guarda el JPEG; se abre al usarlo
faltan = set(corpus.page_id) - set(paginas)
assert not faltan, f"No se encontraron {len(faltan)} páginas en el parquet, por ejemplo {sorted(faltan)[:3]}."
del tabla

def abrir(pid):
    # Imagen (PIL) de una página a partir de su identificador.
    return Image.open(io.BytesIO(paginas[pid])).convert("RGB")
print(f"{len(paginas)} páginas cargadas de {corpus.doc.nunique()} documentos.")

In [ ]:
ejemplo = practica.iloc[0]
print("Pregunta:", ejemplo.pregunta)
print("Respuesta de referencia:", ejemplo.respuesta)
plt.figure(figsize=(6, 8)); plt.imshow(abrir(ejemplo.page_id)); plt.title(ejemplo.page_id); plt.axis("off"); plt.show()

## Modelo que lee las páginas

**Principal: Gemini 3.5 Flash-Lite por API.** No requiere GPU. Creen una clave en [Google AI Studio](https://aistudio.google.com/apikey) y guárdenla en los secretos de Colab (icono de llave en la barra izquierda) con el nombre `GEMINI_API_KEY`. La celda siguiente envía una página y una pregunta para comprobar que la clave funciona.

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

cliente = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

def como_jpeg(img, lado=1600):
    img = img.copy(); img.thumbnail((lado, lado))
    buf = io.BytesIO(); img.save(buf, "JPEG", quality=85)
    return types.Part.from_bytes(data=buf.getvalue(), mime_type="image/jpeg")

r = cliente.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=[como_jpeg(abrir(ejemplo.page_id)), f"Responde en español: {ejemplo.pregunta}"])
print(r.text)

**Alternativa, solo con GPU: Qwen3-VL-2B-Instruct.** Corre dentro de Colab (Entorno de ejecución → Cambiar tipo de entorno → GPU T4). En CPU es demasiado lento para el reto.

In [ ]:
if dispositivo == "cuda":
    from transformers import AutoModelForImageTextToText, AutoProcessor
    qwen = AutoModelForImageTextToText.from_pretrained(
        "Qwen/Qwen3-VL-2B-Instruct", dtype=torch.float16, device_map="cuda").eval()
    proc_qwen = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-2B-Instruct")

    img = abrir(ejemplo.page_id).copy(); img.thumbnail((1280, 1280))
    mensajes = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text", "text": f"Responde en español: {ejemplo.pregunta}"}]}]
    x = proc_qwen.apply_chat_template(mensajes, tokenize=True, add_generation_prompt=True,
                                      return_dict=True, return_tensors="pt").to("cuda")
    y = qwen.generate(**x, max_new_tokens=200, do_sample=False)
    print(proc_qwen.batch_decode(y[:, x["input_ids"].shape[1]:], skip_special_tokens=True)[0])
else:
    print("Sin GPU: usen Gemini.")

## Su copiloto

A partir de aquí, el código es del equipo. Los requisitos mínimos son:

1. Buscar entre las 300 páginas a partir de una pregunta escrita en español.
2. Comparar al menos dos formas de buscar con recall@5 sobre las 20 preguntas de práctica. Una debe ser CLIP (o SigLIP) sobre la imagen de la página; la otra la elige el equipo.
3. Que el VLM lea las páginas encontradas, responda en español y cite el identificador de la página que usó.
4. Que el copiloto se abstenga cuando la respuesta no está en la colección.
5. Una interfaz (Gradio o una celda) que muestre la respuesta y la página citada.

Antes de la demostración, cualquier integrante del equipo debe poder explicar su tabla de recall@5 y por qué eligieron su forma de buscar.

In [ ]:
# Código del equipo